# Citi Bike Automated Data Ingestion

## Purpose
Collect Citi Bike station availability from the GBFS API and append historical snapshots to the Delta table `workspace.citibike.station_status_history`.

## Pipeline
1. Retrieve station status from the Citi Bike GBFS API.
2. Keep operational stations (`is_installed`, `is_renting`, and `is_returning`).
3. Attach a UTC collection timestamp.
4. Calculate station-report freshness.
5. Insert new station/timestamp observations with Delta MERGE.

**Data source:** https://gbfs.citibikenyc.com/gbfs/en/station_status.json

**Note:** This notebook is intended for scheduled ingestion. Station metadata is maintained separately. Validation and exploratory checks are not part of the scheduled path.


## 1. Fetch current station status


In [0]:
import requests

# Citi Bike GBFS station availability endpoint
url = "https://gbfs.citibikenyc.com/gbfs/en/station_status.json"

# Request current station availability data
response = requests.get(url, timeout=20)
response.raise_for_status()

# Extract station records from the API response
stations = response.json()["data"]["stations"]

print("API request successful!")
print("Total stations retrieved:", len(stations))

API request successful!
Total stations retrieved: 2519


## 2. Filter operational stations


In [0]:
# Keep stations that are operational
active_stations = [
    station for station in stations
    if station["is_installed"] == 1
    and station["is_renting"] == 1
    and station["is_returning"] == 1
]

print("Total stations retrieved:", len(stations))
print("Active stations:", len(active_stations))
print("Other stations:", len(stations) - len(active_stations))

Total stations retrieved: 2519
Active stations: 2462
Other stations: 57


## 3. Create a timestamped snapshot


In [0]:
from datetime import datetime, timezone
from pyspark.sql.functions import lit

# Record when this API snapshot was collected
now_utc = datetime.now(timezone.utc)

# Actual collection time
collection_time = now_utc

# Five-minute collection window
collection_window = now_utc.replace(
    minute=(now_utc.minute // 5) * 5,
    second=0,
    microsecond=0
)

# Convert active station records into a Spark DataFrame
df_new_snapshot = (
    spark.createDataFrame(active_stations)
    .withColumn("collected_at", lit(collection_time))
)

print("Collection time (UTC):", collection_time)
print("Station records:", df_new_snapshot.count())

display(
    df_new_snapshot.select(
        "station_id",
        "num_bikes_available",
        "num_docks_available",
        "collected_at"
    ).limit(5)
)

Collection time (UTC): 2026-10-21 18:58:42.260082+00:00
Station records: 2462


station_id,num_bikes_available,num_docks_available,collected_at
2235291403983301624,11,3,2026-10-21T18:58:42.260Z
6969d756-33a6-4a0b-83d0-3db38bc382d1,21,1,2026-10-21T18:58:42.260Z
0630217a-3913-47d9-9540-cf85688296a5,4,14,2026-10-21T18:58:42.260Z
2126623007192357636,9,2,2026-10-21T18:58:42.260Z
66dc3782-0aca-11e7-82f6-3863bb44ef7c,3,24,2026-10-21T18:58:42.260Z


## 4. Prepare freshness fields


In [0]:
from pyspark.sql import functions as F

# Enrich each station snapshot with report timing and freshness information
df_snapshot_ready = (
    df_new_snapshot
    # Convert the station-reported Unix timestamp into a readable timestamp
    .withColumn(
        "station_reported_at",
        F.to_timestamp(F.from_unixtime(F.col("last_reported")))
    )
    # Measure how many minutes old the station report is at collection time
    .withColumn(
        "data_age_minutes",
        (
            F.unix_timestamp("collected_at")
            - F.col("last_reported")
        ) / 60.0
    )
    # Categorize data age: Fresh <= 10 min, Delayed <= 30 min, otherwise Stale
    # Missing timestamps are Unknown; future-dated reports are Invalid
    .withColumn(
        "freshness_status",
        F.when(F.col("data_age_minutes").isNull(), "Unknown")
         .when(F.col("data_age_minutes") < 0, "Invalid")
         .when(F.col("data_age_minutes") <= 10, "Fresh")
         .when(F.col("data_age_minutes") <= 30, "Delayed")
         .otherwise("Stale")
    )
    # Retain only the fields needed in the historical Delta table
    .select(
        "station_id",
        "num_bikes_available",
        "num_docks_available",
        "last_reported",
        "station_reported_at",
        "collected_at",
        "data_age_minutes",
        "freshness_status"
    )
)

# Check the number of prepared records and preview freshness classifications
print("Prepared station records:", df_snapshot_ready.count())

display(
    df_snapshot_ready.select(
        "station_id",
        "data_age_minutes",
        "freshness_status"
    ).limit(5)
)

Prepared station records: 2462


station_id,data_age_minutes,freshness_status
2235291403983301624,5.25,Fresh
6969d756-33a6-4a0b-83d0-3db38bc382d1,2.8,Fresh
0630217a-3913-47d9-9540-cf85688296a5,2.8,Fresh
2126623007192357636,2.783333333333333,Fresh
66dc3782-0aca-11e7-82f6-3863bb44ef7c,2.816666666666667,Fresh


## 5. Merge new observations into Delta history


In [0]:
from delta.tables import DeltaTable

# Open the existing Delta history table; this notebook does not create it
target_table = DeltaTable.forName(
    spark,
    "workspace.citibike.station_status_history"
)

(
    # Compare the incoming snapshot against records already stored
    target_table.alias("target")
    .merge(
        df_snapshot_ready.alias("source"),
        """
        target.station_id = source.station_id
        AND target.collected_at = source.collected_at
        """
    )
    # Insert only station/timestamp pairs not already present (prevents duplicates)
    .whenNotMatchedInsertAll()
    # Apply the merge to the Delta table
    .execute()
)

print("Station availability MERGE completed successfully.")

Station availability MERGE completed successfully.
